In [ ]:
# ============================================================
# build_data_finale.py
# ============================================================
# Ce script :
#   1. Télécharge les données ACEA 2015-2021 (Excel)
#   2. Télécharge les données ACEA 2022-présent (PDFs)
#   3. Fusionne tout dans data/processed/data_finale.csv
#
# Usage : python build_data_finale.py
# ============================================================

In [ ]:
# Bibliothèques

import camelot
import requests
import pandas as pd
from io import BytesIO
import re
import time
import itertools
import datetime
import os

In [ ]:
# ===================== CONFIGURATION =====================
BASE_URL  = "https://www.acea.auto/files/"
HEADERS   = {"User-Agent": "Mozilla/5.0"}
DATA_RAW  = "data/raw"
DATA_PROC = "data/processed"

MOIS_NOMS = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

GROUPES_CIBLES = {
    "Volkswagen Group": "Volkswagen Group",
    "Stellantis":       "Stellantis",
    "Renault Group":    "Renault Group",
    "Toyota Group":     "Toyota Group"
}

GROUP_MAPPING = {
    "renault":    "Renault Group",
    "volkswagen": "Volkswagen Group",
    "toyota":     "Toyota Group",
    "psa":        "Stellantis",
    "fca":        "Stellantis",
    "stellantis": "Stellantis"
}

FALLBACK_COL4 = {
    (2022, 7):  (2023, 7),
    (2022, 8):  (2023, 8),
    (2022, 11): (2023, 11),
    (2022, 12): (2023, 12),
    (2023, 7):  (2024, 7),
    (2023, 9):  (2024, 9),
}

os.makedirs(DATA_RAW,  exist_ok=True)
os.makedirs(DATA_PROC, exist_ok=True)

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE 1 - EXCEL 2015-2021
# ════════════════════════════════════════════════════════════

def identifier_mois(col):
    if hasattr(col, 'month'):
        return col.month
    col_s = str(col).strip()
    m = re.match(r'\d{4}-(\d{2})-\d{2}', col_s)
    if m:
        return int(m.group(1))
    abbrevs = {
        'jan': 1,  'fév': 2, 'fev': 2,  'mar': 3,
        'avr': 4,  'mai': 5, 'jui': 6,  'jul': 7,
        'aoû': 8,  'aou': 8, 'sep': 9,  'oct': 10,
        'nov': 11, 'déc': 12,'dec': 12
    }
    return abbrevs.get(col_s.lower()[:3])


def extraire_excel_2015_2021():
    print("=" * 60)
    print("PARTIE 1 — Extraction Excel 2015-2021")
    print("=" * 60)

    url      = f"{BASE_URL}1990-2021_PC-by-manuf_West-Europe.xlsx"
    response = requests.get(url, headers=HEADERS)
    content  = BytesIO(response.content)

    all_data = []

    for annee in range(2015, 2022):
        df = pd.read_excel(content, sheet_name=str(annee), header=5)
        df["Group"] = df["Group"].ffill()
        df = df[~df["Brand"].str.upper().isin(["TOTAL"])].dropna(subset=["Brand"])

        df["Group_lower"] = df["Group"].str.lower().str.strip()
        mask = df["Group_lower"].apply(lambda x: any(k in x for k in GROUP_MAPPING))
        df   = df[mask].copy()

        def normaliser(g):
            for key, val in GROUP_MAPPING.items():
                if key in g:
                    return val
            return g

        df["group"] = df["Group_lower"].apply(normaliser)

        cols_mois = {}
        for col in df.columns:
            if col in ["Group", "Brand", "Group_lower", "group", "FY"]:
                continue
            mois_num = identifier_mois(col)
            if mois_num:
                cols_mois[col] = mois_num

        df_melt = df[["group", "Brand"] + list(cols_mois.keys())].copy()
        df_melt = df_melt.melt(
            id_vars=["group", "Brand"],
            value_vars=list(cols_mois.keys()),
            var_name="col_mois",
            value_name="registrations"
        )
        df_melt["month_num"] = df_melt["col_mois"].map(cols_mois)
        df_melt["month"]     = df_melt["month_num"].apply(lambda x: MOIS_NOMS[x - 1])
        df_melt["year"]      = annee
        df_melt = df_melt.rename(columns={"Brand": "brand"})
        df_melt = df_melt[["year", "month_num", "month", "group", "brand", "registrations"]]
        df_melt = df_melt.dropna(subset=["registrations"])
        df_melt["registrations"] = df_melt["registrations"].astype(int)

        all_data.append(df_melt)
        print(f"{annee} — {df_melt.shape[0]} lignes")

    df_excel = pd.concat(all_data, ignore_index=True)
    df_excel = df_excel.sort_values(["year", "month_num", "group", "brand"]).reset_index(drop=True)
    print(f"\n===== Excel 2015-2021 : {df_excel.shape[0]} lignes au total =====\n")
    return df_excel

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE 2 - PDFs 2022 → AUJOURD'HUI
# ════════════════════════════════════════════════════════════

def trouver_url_ancien_format(annee_data, mois_data):
    mois_pub  = mois_data + 1 if mois_data < 12 else 1
    annee_pub = annee_data if mois_data < 12 else annee_data + 1
    yymm      = f"{str(annee_data)[2:]}{mois_data:02d}"
    for jour, prefixe, suffixe in itertools.product(
        range(14, 25), ["PRPC", "PCPR"], ["_FINAL", "-FINAL"]
    ):
        date_pub = f"{annee_pub}{mois_pub:02d}{jour:02d}"
        filename = f"{date_pub}_{prefixe}_{yymm}{suffixe}.pdf"
        url      = BASE_URL + filename
        r = requests.head(url, headers=HEADERS, timeout=10, allow_redirects=False)
        if r.status_code == 200 and yymm in filename:
            return url
    return None


def construire_url(annee, mois_num):
    nom = MOIS_NOMS[mois_num - 1]
    if (annee > 2023) or (annee == 2023 and mois_num >= 10):
        if annee == 2024 and mois_num == 8:
            return f"{BASE_URL}Press_release_car_registrations_August-2024.pdf"
        return f"{BASE_URL}Press_release_car_registrations_{nom}_{annee}.pdf"
    return trouver_url_ancien_format(annee, mois_num)


def identifier_table_eu_efta_uk(tables):
    groupes    = list(GROUPES_CIBLES.keys())
    candidates = []
    for i, table in enumerate(tables):
        clean_matches = partial_matches = 0
        for _, row in table.df.iterrows():
            col0 = str(row.iloc[0]).strip()
            if not col0 or col0 == 'nan':
                continue
            for groupe in groupes:
                if col0 == groupe or col0.startswith(groupe + '\n'):
                    clean_matches += 1
                    break
                elif groupe in col0:
                    partial_matches += 1
                    break
        if clean_matches + partial_matches >= 2:
            total_num = sum(
                int(re.sub(r'[^\d]', '', str(v)))
                for v in table.df.values.flatten()
                if re.sub(r'[^\d]', '', str(v))
            )
            candidates.append((i, table, clean_matches, total_num))
    if not candidates:
        return None, None
    candidates.sort(key=lambda x: (x[2], x[3]), reverse=True)
    idx, table, _, _ = candidates[0]
    return idx, table


def nettoyer_brand(s):
    return re.sub(r'\d+$', '', s).strip()

def nettoyer_valeur(s):
    clean = re.sub(r'[^\d]', '', str(s))
    return int(clean) if clean else None

def extraire_groupe_depuis_col0(col0):
    parts = [p.strip() for p in col0.split('\n') if p.strip()]
    for part in reversed(parts):
        if part in GROUPES_CIBLES:
            return GROUPES_CIBLES[part]
    return None

def parse_inline(names, values, current_group, results):
    for name, val in zip(names, values):
        name = name.strip()
        if not name:
            continue
        if name in GROUPES_CIBLES:
            current_group = GROUPES_CIBLES[name]
            continue
        if name.lower().endswith("group") and name not in GROUPES_CIBLES:
            current_group = None
            continue
        if current_group is None:
            continue
        if re.match(r'^[\d.,+\-\s]+$', name):
            continue
        brand = nettoyer_brand(name)
        if not brand or brand.lower() == 'total':
            continue
        registrations = nettoyer_valeur(val)
        if registrations:
            results.append({
                "group":         current_group,
                "brand":         brand,
                "registrations": registrations
            })
    return current_group

def parse_tableau(table_df, col_valeurs=3):
    results       = []
    current_group = None
    rows          = list(table_df.iterrows())
    for i, (_, row) in enumerate(rows):
        nb_cols = len(row)
        col0    = str(row.iloc[0]).strip()
        col_val = str(row.iloc[col_valeurs]).strip() if nb_cols > col_valeurs else ''
        if not col0 or col0 == 'nan':
            continue
        mots_cles = [
            'DECEMBER','JANUARY','FEBRUARY','MARCH','APRIL','MAY',
            'JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER',
            'UNITS','SHARE','JAN-','JUL-','APR-'
        ]
        if any(h in col0.upper() for h in mots_cles):
            continue
        names  = [n.strip() for n in col0.split('\n') if n.strip()]
        values = [v.strip() for v in col_val.split('\n') if v.strip()]
        first  = names[0]
        if first in GROUPES_CIBLES:
            current_group = GROUPES_CIBLES[first]
            if len(names) > 1:
                brand_names  = names[1:]
                brand_values = values[1:] if len(values) > 1 else []
                if not brand_values and i + 1 < len(rows):
                    _, next_row = rows[i + 1]
                    next_col0   = str(next_row.iloc[0]).strip()
                    next_col_v  = str(next_row.iloc[col_valeurs]).strip() if len(next_row) > col_valeurs else ''
                    if not next_col0 or next_col0 == 'nan':
                        brand_values = [v.strip() for v in next_col_v.split('\n') if v.strip()]
                current_group = parse_inline(brand_names, brand_values, current_group, results)
        else:
            groupe_detecte = extraire_groupe_depuis_col0(col0)
            if groupe_detecte:
                current_group = groupe_detecte
                brand_names = [
                    n for n in names
                    if not re.match(r'^[\d.,+\-\s]+$', n)
                    and n not in GROUPES_CIBLES
                    and not (n.lower().endswith("group") and n not in GROUPES_CIBLES)
                ]
                current_group = parse_inline(brand_names, values, current_group, results)
            elif current_group is not None:
                current_group = parse_inline(names, values, current_group, results)
    return pd.DataFrame(results)


def telecharger_pdf(url, pdf_path):
    if os.path.exists(pdf_path):
        return True
    response = requests.get(url, headers=HEADERS, timeout=30)
    if response.status_code != 200:
        return False
    with open(pdf_path, "wb") as f:
        f.write(response.content)
    return True

def extraire_donnees(pdf_path, col_valeurs=3):
    try:
        tables = camelot.read_pdf(pdf_path, pages="all", flavor="lattice")
    except Exception as e:
        return None, f"camelot: {e}"
    idx, table = identifier_table_eu_efta_uk(tables)
    if table is None:
        return None, "table non détectée"
    df = parse_tableau(table.df, col_valeurs=col_valeurs)
    if df.empty:
        return None, "dataframe vide"
    return df, None


def extraire_pdf_2022_present():
    print("=" * 60)
    print("PARTIE 2 - Extraction des PDFs 2022 --> Aujourd'hui")
    print("=" * 60)

    aujourd_hui  = datetime.date.today()
    annee_limite = aujourd_hui.year
    mois_limite  = aujourd_hui.month - 1 or 12

    all_data = []
    erreurs  = []

    for annee in range(2022, annee_limite + 1):
        for mois_num in range(1, 13):

            if annee == annee_limite and mois_num > mois_limite:
                break
            if annee > annee_limite:
                break

            mois_nom = MOIS_NOMS[mois_num - 1]
            print(f"\n----{mois_nom} {annee}")

            # Stratégie A : fallback col4
            if (annee, mois_num) in FALLBACK_COL4:
                src_annee, src_mois = FALLBACK_COL4[(annee, mois_num)]
                src_nom = MOIS_NOMS[src_mois - 1]
                url     = construire_url(src_annee, src_mois)
                pdf_src = f"{DATA_RAW}/acea_{src_nom}_{src_annee}.pdf"
                if url and telecharger_pdf(url, pdf_src):
                    df, err = extraire_donnees(pdf_src, col_valeurs=4)
                    if df is not None:
                        df["month"]     = mois_nom
                        df["month_num"] = mois_num
                        df["year"]      = annee
                        all_data.append(df)
                        print(f"{len(df)} lignes — {df['group'].nunique()} groupes [col4 fallback]")
                        time.sleep(0.5)
                        continue
                    print(f"Fallback col4 échoué : {err}")
                else:
                    print(f"Fallback col4 : PDF source introuvable")

            # Stratégie B : URL directe
            try:
                url = construire_url(annee, mois_num)
                if not url:
                    print(f"URL introuvable")
                    erreurs.append({"mois": mois_nom, "annee": annee, "raison": "URL non trouvée"})
                    continue

                print(f"{url.split('/')[-1]}")
                pdf_path = f"{DATA_RAW}/acea_{mois_nom}_{annee}.pdf"

                if not telecharger_pdf(url, pdf_path):
                    print(f"-----Téléchargement échoué-----")
                    erreurs.append({"mois": mois_nom, "annee": annee, "raison": "téléchargement échoué"})
                    continue

                df, err = extraire_donnees(pdf_path, col_valeurs=3)
                if err:
                    print(f" {err}")
                    erreurs.append({"mois": mois_nom, "annee": annee, "raison": err})
                    continue

                df["month"]     = mois_nom
                df["month_num"] = mois_num
                df["year"]      = annee
                all_data.append(df)
                print(f"{len(df)} lignes — {df['group'].nunique()} groupes")

            except Exception as e:
                print(f" Erreur : {e}")
                erreurs.append({"mois": mois_nom, "annee": annee, "raison": str(e)})

            time.sleep(0.5)

    if all_data:
        df_pdf = pd.concat(all_data, ignore_index=True)
        df_pdf = df_pdf[["year", "month_num", "month", "group", "brand", "registrations"]]
        df_pdf = df_pdf.sort_values(["year", "month_num", "group", "brand"]).reset_index(drop=True)
        print(f"\n------- PDFs 2022-présent : {df_pdf.shape[0]} lignes ---------")
    else:
        df_pdf = pd.DataFrame(columns=["year", "month_num", "month", "group", "brand", "registrations"])

    if erreurs:
        print(f"\n{len(erreurs)} erreurs :")
        print(pd.DataFrame(erreurs).to_string(index=False))

    return df_pdf

In [ ]:
# ════════════════════════════════════════════════════════════
# PARTIE 3 - CONSOLIDATION DONNEES 2015-2021 & 2022-2026
# ════════════════════════════════════════════════════════════

def consolider(df_excel, df_pdf):
    print("\n" + "=" * 60)
    print("PARTIE 3 - Consolidation finale")
    print("=" * 60)

    cols = ["year", "month_num", "month", "group", "brand", "registrations"]

    data_finale = pd.concat(
        [df_excel[cols], df_pdf[cols]],
        ignore_index=True
    ).sort_values(
        ["year", "month_num", "group", "brand"]
    ).reset_index(drop=True)

    path = f"{DATA_PROC}/data_finale.csv"
    data_finale.to_csv(path, index=False)

    print(f"\n==== data_finale.csv créé avec succès ====")
    print(f"   Lignes   : {data_finale.shape[0]}")
    print(f"   Période  : {data_finale['year'].min()} → {data_finale['year'].max()}")
    print(f"   Groupes  : {sorted(data_finale['group'].unique())}")
    print(f"   Fichier  : {path}")

    return data_finale

In [ ]:
# ════════════════════════════════════════════════════════════
# LANCEMENT
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    df_excel    = extraire_excel_2015_2021()
    df_pdf      = extraire_pdf_2022_present()
    data_finale = consolider(df_excel, df_pdf)